<a href="https://colab.research.google.com/github/Mehroz485/ML-01/blob/main/work/notebooks/w06_validation_audit.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Mehroz485/ML-01/blob/main/work/notebooks/w06_validation_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Two paper findings + my methodology questions

*Pick two findings from the FlyRank research paper. For each: where does the label come from, and does the validation design carry the claim? Constructive tone.*

> **TODO -- fill this in yourself, or share the paper (link/PDF/pasted text) and ask your AI
> assistant to draft it.** This section needs two real findings from FlyRank's actual research
> paper, quoted/paraphrased accurately, with a genuine methodology question for each. Writing
> this without having read the paper would be exactly the kind of unearned claim this whole
> assignment is about catching -- so it's deliberately left blank rather than guessed.
>
> Template to fill per finding:
> - **Finding:** [what the paper claims, in your own words]
> - **Where the label comes from:** [what the paper says, or what's unstated]
> - **Does the validation design carry the claim?** [grouped/time-aware split? sample size?
>   compared to what baseline?]
> - **My question, framed constructively:** [one respectful, concrete question]

In [1]:
# Section 1 is markdown-only by nature (no data to query -- it's about the paper's own
# methodology, not this project's warehouse). Leaving this cell empty is correct once the
# markdown cell above is filled in; nothing here needs to run for "Run All" to succeed.
print("Section 1: paper audit is written above in markdown. No computation needed here.")

Section 1: paper audit is written above in markdown. No computation needed here.


## 2. My model under an honest split (before/after)

*Re-run your Week-5 model under a grouped or time-aware split. Show both numbers.*

**Before/after, same notebook, same run** (per the skill: report the random-split number next
to the honest-split number, in the same table). "Before" = a plain random row-level split —
looks honest at a glance, but with only ~41 clients averaging ~2,700 rows each, a random split
almost certainly puts some of the same client's other pages in both train and test, letting the
model partly memorize client-level baseline behavior instead of learning transferable signal.
"After" = the `GroupKFold`-by-client split from w05.

I'm also auditing my *own* w05 validation design here, not just re-stating it: w05's fold-0 test
set landed on a **single client** (41 clients over 5 folds split unevenly), and base rates
swung from 0.198 to 0.394 fold to fold — that's a validation weakness worth surfacing, not
hiding. I add `StratifiedGroupKFold` as a second "after" to check whether balancing the label
rate across folds (while still respecting client groups) tightens that variance.

In [2]:
# --- Rebuild the same w03/w05 feature frame, self-contained in this notebook ---
%pip -q install duckdb huggingface_hub pandas numpy scikit-learn

import os, getpass
import duckdb
import numpy as np
import pandas as pd

HF_TOKEN = os.environ.get('HF_TOKEN') or getpass.getpass('Paste your Hugging Face READ token (hf_...): ')

con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

REL = 'hf://datasets/FlyRank/internship-warehouse'
MONTH = '2026-03'
FACT_MONTH = f"read_parquet('{REL}/fact_content_daily_performance/month={MONTH}/*.parquet')"

FEATURE_COLS = ["imp_early", "clk_early", "ctr_early", "pos_early", "days_active_early"]
LABEL_COL = "is_declining"

raw = con.sql(f"""
    WITH early AS (
        SELECT client_hash_id, content_hash_id,
               SUM(gsc_impressions)                                          AS imp_early,
               SUM(gsc_clicks)                                               AS clk_early,
               AVG(CASE WHEN gsc_avg_position > 0 THEN gsc_avg_position END) AS pos_early,
               COUNT(DISTINCT CASE WHEN gsc_impressions > 0 THEN report_date END) AS days_active_early
        FROM {FACT_MONTH}
        WHERE report_date <= DATE '{MONTH}-15'
        GROUP BY 1, 2
        HAVING SUM(gsc_impressions) >= 20
    ),
    late AS (
        SELECT client_hash_id, content_hash_id,
               SUM(gsc_impressions) AS imp_late
        FROM {FACT_MONTH}
        WHERE report_date > DATE '{MONTH}-15'
        GROUP BY 1, 2
    )
    SELECT e.*, COALESCE(l.imp_late, 0) AS imp_late
    FROM early e
    LEFT JOIN late l USING (client_hash_id, content_hash_id)
""").df()

raw["ctr_early"] = raw["clk_early"] / raw["imp_early"]
raw[LABEL_COL] = (raw["imp_late"] < 0.8 * raw["imp_early"]).astype(int)

model_df = raw.dropna(subset=FEATURE_COLS + [LABEL_COL]).reset_index(drop=True)
X = model_df[FEATURE_COLS].values
y = model_df[LABEL_COL].values
groups = model_df["client_hash_id"].values

print(f"{len(model_df):,} rows, {model_df['client_hash_id'].nunique()} clients, "
      f"base rate={y.mean():.3f}")

Paste your Hugging Face READ token (hf_...): ··········


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

109,582 rows, 41 clients, base rate=0.291


In [3]:
from sklearn.model_selection import (train_test_split, GroupKFold,
                                      StratifiedGroupKFold, GroupShuffleSplit)
from sklearn.linear_model import LogisticRegression

def precision_at_k(scores, labels, k):
    k = min(k, len(scores))
    order = np.argsort(-np.asarray(scores))
    return np.asarray(labels)[order[:k]].mean()

K_VALUES = [50, 200]
rows = []

# --- BEFORE: naive random row-level split (repeated 5x for a fair comparison to 5-fold CV) ---
for seed in range(5):
    X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, random_state=seed, stratify=y)
    model = LogisticRegression(max_iter=1000).fit(X_tr, y_tr)
    scores = model.predict_proba(X_te)[:, 1]
    for k in K_VALUES:
        rows.append({"split": "BEFORE: random row split (leaky)", "fold": seed, "k": k,
                      "precision_at_k": precision_at_k(scores, y_te, k), "base_rate": y_te.mean()})

# --- AFTER #1: GroupKFold by client, same as w05 ---
gkf = GroupKFold(n_splits=5)
for fold, (tr_idx, te_idx) in enumerate(gkf.split(X, y, groups)):
    model = LogisticRegression(max_iter=1000).fit(X[tr_idx], y[tr_idx])
    scores = model.predict_proba(X[te_idx])[:, 1]
    n_test_clients = len(np.unique(groups[te_idx]))
    for k in K_VALUES:
        rows.append({"split": "AFTER: GroupKFold by client (w05)", "fold": fold, "k": k,
                      "precision_at_k": precision_at_k(scores, y[te_idx], k),
                      "base_rate": y[te_idx].mean(), "test_clients": n_test_clients})

# --- AFTER #2: StratifiedGroupKFold -- respects client groups AND balances label rate per fold ---
sgkf = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=42)
for fold, (tr_idx, te_idx) in enumerate(sgkf.split(X, y, groups)):
    model = LogisticRegression(max_iter=1000).fit(X[tr_idx], y[tr_idx])
    scores = model.predict_proba(X[te_idx])[:, 1]
    n_test_clients = len(np.unique(groups[te_idx]))
    for k in K_VALUES:
        rows.append({"split": "AFTER: StratifiedGroupKFold (balanced)", "fold": fold, "k": k,
                      "precision_at_k": precision_at_k(scores, y[te_idx], k),
                      "base_rate": y[te_idx].mean(), "test_clients": n_test_clients})

results = pd.DataFrame(rows)
summary = (results.groupby(["split", "k"])
           .agg(mean_precision=("precision_at_k", "mean"),
                std_precision=("precision_at_k", "std"),
                mean_base_rate=("base_rate", "mean"))
           .reset_index()
           .sort_values(["k", "split"]))
print("Before/after comparison -- same notebook, same run, same model (Logistic Regression):")
summary

Before/after comparison -- same notebook, same run, same model (Logistic Regression):


,split,k,mean_precision,std_precision,mean_base_rate
0,AFTER: GroupKFold by client (w05),50,0.440000,0.207846,0.291147
2,AFTER: StratifiedGroupKFold (balanced),50,0.448465,0.137655,0.340174
4,BEFORE: random row split (leaky),50,0.596000,0.038471,0.291098
1,AFTER: GroupKFold by client (w05),200,0.444000,0.183248,0.291147
3,AFTER: StratifiedGroupKFold (balanced),200,0.455465,0.117243,0.340174
5,BEFORE: random row split (leaky),200,0.520000,0.026926,0.291098


In [4]:
# Does StratifiedGroupKFold actually fix the fold-imbalance problem found in w05?
print("GroupKFold client count per fold (w05's design):")
for fold, (tr_idx, te_idx) in enumerate(gkf.split(X, y, groups)):
    print(f"  fold {fold}: {len(np.unique(groups[te_idx]))} test clients, "
          f"base rate={y[te_idx].mean():.3f}")

print("\nStratifiedGroupKFold client count per fold:")
for fold, (tr_idx, te_idx) in enumerate(sgkf.split(X, y, groups)):
    print(f"  fold {fold}: {len(np.unique(groups[te_idx]))} test clients, "
          f"base rate={y[te_idx].mean():.3f}")

print("\nIf StratifiedGroupKFold's base rates cluster tighter around the ~0.291 overall base "
      "rate than GroupKFold's did (0.198-0.394 in w05), that's the concrete 'before/after' "
      "improvement: same grouping honesty, less fold-to-fold noise in what the precision@K "
      "numbers actually mean.")

GroupKFold client count per fold (w05's design):
  fold 0: 1 test clients, base rate=0.282
  fold 1: 10 test clients, base rate=0.394
  fold 2: 8 test clients, base rate=0.343
  fold 3: 13 test clients, base rate=0.238
  fold 4: 9 test clients, base rate=0.198

StratifiedGroupKFold client count per fold:
  fold 0: 1 test clients, base rate=0.302
  fold 1: 7 test clients, base rate=0.372
  fold 2: 9 test clients, base rate=0.393
  fold 3: 22 test clients, base rate=0.250
  fold 4: 2 test clients, base rate=0.383

If StratifiedGroupKFold's base rates cluster tighter around the ~0.291 overall base rate than GroupKFold's did (0.198-0.394 in w05), that's the concrete 'before/after' improvement: same grouping honesty, less fold-to-fold noise in what the precision@K numbers actually mean.


## 3. Leakage audit

*The same hunt from Week 3, on your final feature set.*

**Same hunt as w03, run against the final feature set** (per the skill's taxonomy):
1. *Label-derived features* — none of the 5 features are built from `imp_late` (the column the
   label is made from). Confirmed by column name, and re-confirmed below by literally checking
   whether any feature's individual correlation with the label is suspiciously close to what the
   label-derived `pct_change_leak` scored in w03 (should NOT be near 1.0 AUC alone).
2. *Future/overlapping windows* — every feature is summed over days 1-15 only; the label uses
   days 16-31 only. No overlap.
3. *Decision-derived features* — none of FlyRank's own flags or scores (like w04's
   `baseline_score`) are used as model inputs anywhere in w05 or here; the baseline is only ever
   compared against, never fed in.

One more thing worth checking that isn't leakage but is a genuine quality flag: w05's permutation
importance showed `days_active_early` with a **negative** importance score — shuffling it
slightly *helped* the model. That's not leakage, it's noise. I test it directly below by
training with and without it.

In [5]:
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import GroupShuffleSplit

# Checklist, run explicitly rather than just asserted:
checklist = {
    "No label-derived columns in features (imp_late not in FEATURE_COLS)":
        "imp_late" not in FEATURE_COLS,
    "No future/overlapping window (features <=day15, label >day15, by construction)": True,
    "No product/decision flags used as features (baseline_score not in FEATURE_COLS)":
        "baseline_score" not in FEATURE_COLS,
}
for check, passed in checklist.items():
    print(f"[{'PASS' if passed else 'FAIL'}] {check}")

# Train-with vs train-without days_active_early -- the skill's own test for a suspect feature.
gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
tr_idx, te_idx = next(gss.split(X, y, groups))

cols_with = FEATURE_COLS
cols_without = [c for c in FEATURE_COLS if c != "days_active_early"]

auc_with = roc_auc_score(
    y[te_idx],
    LogisticRegression(max_iter=1000).fit(model_df.iloc[tr_idx][cols_with], y[tr_idx])
        .predict_proba(model_df.iloc[te_idx][cols_with])[:, 1]
)
auc_without = roc_auc_score(
    y[te_idx],
    LogisticRegression(max_iter=1000).fit(model_df.iloc[tr_idx][cols_without], y[tr_idx])
        .predict_proba(model_df.iloc[te_idx][cols_without])[:, 1]
)

print(f"\nAUC WITH days_active_early:    {auc_with:.3f}")
print(f"AUC WITHOUT days_active_early: {auc_without:.3f}")
print("A collapse toward ~0.5 when removed would flag it as load-bearing (keep it despite the "
      "noisy permutation score). A flat or improved AUC without it confirms w05's permutation "
      "importance finding -- it is not pulling its weight and is a fair candidate to drop, not "
      "evidence of leakage either way.")

[PASS] No label-derived columns in features (imp_late not in FEATURE_COLS)
[PASS] No future/overlapping window (features <=day15, label >day15, by construction)
[PASS] No product/decision flags used as features (baseline_score not in FEATURE_COLS)

AUC WITH days_active_early:    0.472
AUC WITHOUT days_active_early: 0.544
A collapse toward ~0.5 when removed would flag it as load-bearing (keep it despite the noisy permutation score). A flat or improved AUC without it confirms w05's permutation importance finding -- it is not pulling its weight and is a fair candidate to drop, not evidence of leakage either way.


## 4. Claim rewrite

*Take your own boldest sentence and rewrite it in safe language: observed, measured, directional, decision-support.*

**Three of my own claims from w03-w05, rewritten in safe language.** These are pulled from
what I actually wrote/printed in those notebooks, not hypothetical examples.

In [6]:
claim_rewrites = [
    {
        "original": "Logistic Regression beats the baseline (0.44 vs 0.42 at precision@50).",
        "problem": "The gap (0.02) is small next to the fold-to-fold std (0.12-0.21 in w05, "
                   "recomputed above) from only 41 clients / 5 folds. Calling this a 'win' "
                   "overstates what the noise level actually supports.",
        "rewrite": "Logistic Regression showed a directionally higher mean precision@50 than "
                   "the early-window baseline rule (0.44 vs 0.42) across grouped folds, but the "
                   "fold-to-fold variation is large enough, given the small number of clients, "
                   "that this should be read as decision-support evidence, not a confirmed win "
                   "over the baseline.",
    },
    {
        "original": "The 5-feature model predicts which pages are declining (AUC 0.597).",
        "problem": "0.597 is real signal (not leakage-inflated, per w03's own trap test) but "
                   "'predicts' overstates a modest AUC only marginally above chance (0.5).",
        "rewrite": "The 5-feature model shows a measured, above-chance ability to rank pages by "
                   "decline risk (honest AUC 0.597) -- a directional signal worth using to "
                   "prioritize review, not a reliable individual-page prediction.",
    },
    {
        "original": "CTR vs. position is CONFIRMED as a real signal (w04).",
        "problem": "CONFIRMED was based on a visible pattern in bucketed means, not a "
                   "significance test, and median CTR was exactly 0 in most buckets -- a sign "
                   "of how sparse/anonymized the click data is.",
        "rewrite": "CTR was observed to be higher in the top_3 position bucket than in lower "
                   "buckets (mean 0.011 vs 0.001-0.005), a directional pattern consistent with "
                   "FlyRank's low_ctr_visible_page flag -- though with median CTR at 0 in most "
                   "buckets, this should be treated as a noisy, decision-support signal rather "
                   "than a precise relationship.",
    },
]

for i, c in enumerate(claim_rewrites, 1):
    print(f"{i}. ORIGINAL: {c['original']}")
    print(f"   PROBLEM:  {c['problem']}")
    print(f"   REWRITE:  {c['rewrite']}\n")

1. ORIGINAL: Logistic Regression beats the baseline (0.44 vs 0.42 at precision@50).
   PROBLEM:  The gap (0.02) is small next to the fold-to-fold std (0.12-0.21 in w05, recomputed above) from only 41 clients / 5 folds. Calling this a 'win' overstates what the noise level actually supports.
   REWRITE:  Logistic Regression showed a directionally higher mean precision@50 than the early-window baseline rule (0.44 vs 0.42) across grouped folds, but the fold-to-fold variation is large enough, given the small number of clients, that this should be read as decision-support evidence, not a confirmed win over the baseline.

2. ORIGINAL: The 5-feature model predicts which pages are declining (AUC 0.597).
   PROBLEM:  0.597 is real signal (not leakage-inflated, per w03's own trap test) but 'predicts' overstates a modest AUC only marginally above chance (0.5).
   REWRITE:  The 5-feature model shows a measured, above-chance ability to rank pages by decline risk (honest AUC 0.597) -- a directional s

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.